# Gesture Recognition - Model Deployment

This notebook handles model pruning, quantization, and ONNX export for deployment.

**Optimization Pipeline:**
```
Original Model (FP32, 87K params)
    ↓ Structured Pruning (30% channels)
Pruned Model (FP32, 46K params) + Fine-tuning
    ↓ ONNX Export
    ↓ ONNX Runtime INT8 Quantization
Deployable ONNX Model
```

**Output:** ONNX model for Android deployment

## 1. Environment Setup

In [1]:
# Check GPU
!nvidia-smi
# Install dependencies
!pip install -q onnx onnxruntime onnxscript scikit-learn tqdm

NVIDIA-SMI has failed because it couldn't communicate with the NVIDIA driver. Make sure that the latest NVIDIA driver is installed and running.

^C
ERROR: Operation cancelled by user


In [8]:
import os
import json
import copy
import time
from collections import defaultdict

import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from common.constants import *
from common.log import log_warn, log_err, log_info
from common.utils import to_scalar, resample, to_raw_sequence, compute_features
from common.gesture_tcn_model import GestureTCN

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cpu


## 2. Mount Google Drive and Load Dataset Info

In [9]:
""" Load raw dataset from google drive
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
"""

' Load raw dataset from google drive\n# Mount Google Drive\nfrom google.colab import drive\ndrive.mount("/content/drive")\n'

In [10]:
# Paths
save_dir = "checkpoints"
info_path = os.path.join(save_dir, "dataset_info.json")

with open(info_path, "r", encoding="utf-8") as f:
    dataset_info = json.load(f)

# Set constants
SEQ_LEN = dataset_info["seq_len"]
FEATURE_DIM = dataset_info["feature_dim"]
NUM_CLASSES = dataset_info["num_classes"]
CLASS_NAMES = dataset_info["class_names"]
RAW_DIM = dataset_info["raw_dim"]
CACHE_VERSION = dataset_info["cache_version"]
NUM_LANDMARKS = 21
NUM_COORDS = 3
WRIST_IDX = 0
MID_FINGER_IDX = 9
PAIRS = [tuple(p) for p in dataset_info["pairs"]]
FINGER_CHAINS = dataset_info["finger_chains"]
N_FINGERS = dataset_info["n_fingers"]
N_PAIRS = len(PAIRS)

# Pruning parameter
PRUNE_RATIO = 0.3

print(f"Feature dim: {FEATURE_DIM}")
print(f"Classes: {CLASS_NAMES}")
print(f"Prune ratio: {PRUNE_RATIO}")

Feature dim: 144
Classes: ['grab', 'release', 'swipe_up', 'swipe_down', 'noise']
Prune ratio: 0.3


## 3. Model Definition

In [11]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def get_model_size_mb(model):
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024**2)

## 4. Load Dataset for Fine-tuning and Calibration

In [12]:
def compute_features(raw_seq):
    raw_seq = np.asarray(raw_seq, dtype=np.float32)
    if raw_seq.ndim != 2 or raw_seq.shape[1] != RAW_DIM:
        if raw_seq.ndim == 1:
            raw_seq = raw_seq.reshape(-1, RAW_DIM)

    T = raw_seq.shape[0]
    if T == 0:
        return np.zeros((0, FEATURE_DIM), dtype=np.float32)

    lms = raw_seq.reshape(T, NUM_LANDMARKS, NUM_COORDS)
    wrist = lms[:, WRIST_IDX, :]
    relative = lms - wrist[:, np.newaxis, :]
    mid = lms[:, MID_FINGER_IDX, :]
    palm_size = np.maximum(np.linalg.norm(mid - wrist, axis=-1, keepdims=True), 1e-6)
    norm_lms = relative / palm_size[:, np.newaxis, :]
    norm_flat = norm_lms.reshape(T, -1).astype(np.float32)

    vel = np.zeros_like(norm_flat)
    if T > 1:
        vel[1:] = norm_flat[1:] - norm_flat[:-1]
        vel[0] = vel[1]

    wrist_vel = np.zeros((T, 3), dtype=np.float32)
    if T > 1:
        wrist_vel[1:] = wrist[1:] - wrist[:-1]
        wrist_vel[0] = wrist_vel[1]

    dists = np.zeros((T, N_PAIRS), dtype=np.float32)
    for k, (i, j) in enumerate(PAIRS):
        dists[:, k] = np.linalg.norm(norm_lms[:, i] - norm_lms[:, j], axis=-1)

    angles = np.zeros((T, N_FINGERS), dtype=np.float32)
    for fi, chain in enumerate(FINGER_CHAINS):
        v1 = lms[:, chain[1]] - lms[:, chain[0]]
        v2 = lms[:, chain[-1]] - lms[:, chain[1]]
        n1 = np.linalg.norm(v1, axis=-1, keepdims=True) + 1e-8
        n2 = np.linalg.norm(v2, axis=-1, keepdims=True) + 1e-8
        cos_a = np.clip((v1 / n1 * v2 / n2).sum(-1), -1.0, 1.0)
        angles[:, fi] = np.arccos(cos_a)

    feat = np.concatenate([norm_flat, vel, wrist_vel, dists, angles], axis=1)
    return feat.astype(np.float32)


class GestureDataset(Dataset):
    def __init__(self, cache_path, norm_stats=None):
        data = np.load(cache_path, allow_pickle=True)
        self.samples = data["samples"]
        self.labels = data["labels"]
        self.norm_stats = norm_stats

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        raw = self.samples[idx]
        feat = compute_features(raw)
        x = torch.FloatTensor(feat.T)
        if self.norm_stats is not None:
            mean = torch.FloatTensor(self.norm_stats["mean"]).unsqueeze(1)
            std = torch.FloatTensor(self.norm_stats["std"]).unsqueeze(1)
            x = (x - mean) / (std + 1e-8)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

In [13]:
# Load normalization stats
norm_stats = torch.load(os.path.join(save_dir, "norm_stats.pt"), weights_only=False)

# Load datasets
cache_dir = os.path.join(save_dir, "cache")
train_cache = os.path.join(cache_dir, f"train_{CACHE_VERSION}.npz")
test_cache = os.path.join(cache_dir, f"test_{CACHE_VERSION}.npz")

train_dataset = GestureDataset(train_cache, norm_stats)
test_dataset = GestureDataset(test_cache, norm_stats)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Train samples: 3018
Test samples: 29


In [14]:
# Load trained model
original_model = GestureTCN().to(DEVICE)
ckpt_path = os.path.join(save_dir, "gesture_tcn_best.pth")
original_model.load_state_dict(
    torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
)
original_model.eval()

print("=== Original Model ===")
print(f"Parameters: {count_parameters(original_model):,}")
print(f"Size: {get_model_size_mb(original_model):.4f} MB")

=== Original Model ===
Parameters: 87,077
Size: 0.3365 MB


## 6. Structured Pruning

Structured pruning removes entire channels/filters, resulting in real parameter reduction.

In [15]:
def create_pruned_model(prune_ratio=0.3):
    """Create a new model with reduced channel count."""

    def round_to_8(x):
        # Round to multiple of 8 for optimal SIMD performance
        return max(8, round(x / 8) * 8)

    channels = {
        "stem": round_to_8(int(48 * (1 - prune_ratio))),
        "mid": round_to_8(int(48 * (1 - prune_ratio))),
        "out": round_to_8(int(64 * (1 - prune_ratio))),
        "head": round_to_8(int(32 * (1 - prune_ratio))),
    }

    print(f"Pruned channels: {channels}")
    return GestureTCN(channels=channels).to(DEVICE), channels


log_info(
    f"Creating Structurally Pruned Model ({PRUNE_RATIO * 100:.0f}% channels removed)"
)
pruned_model, pruned_channels = create_pruned_model(PRUNE_RATIO)

print(f"\nOriginal parameters: {count_parameters(original_model):,}")
print(f"Pruned parameters: {count_parameters(pruned_model):,}")
print(
    f"Compression: {count_parameters(original_model) / count_parameters(pruned_model):.2f}x"
)
print(f"Pruned size: {get_model_size_mb(pruned_model):.4f} MB")

[22:34:36] INFO - Creating Structurally Pruned Model (30% channels removed)
Pruned channels: {'stem': 32, 'mid': 32, 'out': 48, 'head': 24}

Original parameters: 87,077
Pruned parameters: 45,877
Compression: 1.90x
Pruned size: 0.1781 MB


## 7. Fine-tuning Pruned Model

In [16]:
@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    acc = (all_preds == all_labels).mean()
    f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    return {
        "accuracy": acc,
        "f1_score": f1,
        "predictions": all_preds,
        "labels": all_labels,
    }

In [17]:
def fine_tune_model(model, train_loader, test_loader, epochs=100, lr=1e-3):
    """Fine-tune pruned model to recover accuracy."""
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-5
    )

    best_acc = 0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()

            train_loss += loss.item() * x.size(0)
            train_correct += (logits.argmax(1) == y).sum().item()
            train_total += x.size(0)

        scheduler.step()
        metrics = evaluate_model(model, test_loader, DEVICE)

        if metrics["accuracy"] > best_acc:
            best_acc = metrics["accuracy"]
            best_state = copy.deepcopy(model.state_dict())

        if epoch % 10 == 0 or epoch == 1:
            log_info(
                f"Epoch {epoch:3d} | Loss: {train_loss / train_total:.4f} | "
                f"Train Acc: {train_correct / train_total:.4f} | Test Acc: {metrics['accuracy']:.4f}"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_acc

In [19]:
log_info("Fine-tuning Pruned Model ...")
pruned_model, pruned_best_acc = fine_tune_model(
    pruned_model, train_loader, test_loader, epochs=100
)
log_info(f"Best accuracy after fine-tuning: {pruned_best_acc:.4f}")

[22:35:08] INFO - Fine-tuning Pruned Model ...
[22:35:11] INFO - Epoch   1 | Loss: 0.0565 | Train Acc: 0.9844 | Test Acc: 0.7931
[22:35:41] INFO - Epoch  10 | Loss: 0.0213 | Train Acc: 0.9957 | Test Acc: 0.8276
[22:36:12] INFO - Epoch  20 | Loss: 0.0123 | Train Acc: 0.9964 | Test Acc: 0.8276
[22:36:44] INFO - Epoch  30 | Loss: 0.0075 | Train Acc: 0.9980 | Test Acc: 0.7931
[22:37:17] INFO - Epoch  40 | Loss: 0.0022 | Train Acc: 0.9993 | Test Acc: 0.7931
[22:37:50] INFO - Epoch  50 | Loss: 0.0012 | Train Acc: 0.9997 | Test Acc: 0.8276
[22:38:20] INFO - Epoch  60 | Loss: 0.0017 | Train Acc: 0.9993 | Test Acc: 0.7931
[22:38:52] INFO - Epoch  70 | Loss: 0.0011 | Train Acc: 0.9993 | Test Acc: 0.7931
[22:39:23] INFO - Epoch  80 | Loss: 0.0004 | Train Acc: 1.0000 | Test Acc: 0.7931
[22:39:58] INFO - Epoch  90 | Loss: 0.0001 | Train Acc: 1.0000 | Test Acc: 0.7931
[22:40:32] INFO - Epoch 100 | Loss: 0.0002 | Train Acc: 1.0000 | Test Acc: 0.7931
[22:40:32] INFO - Best accuracy after fine-tuning: 

In [20]:
# Evaluate pruned model
pruned_metrics = evaluate_model(pruned_model, test_loader, DEVICE)
print(f"\n=== Pruned Model Evaluation ===")
print(f"Accuracy: {pruned_metrics['accuracy']:.4f}")
print(f"F1-Score: {pruned_metrics['f1_score']:.4f}")


=== Pruned Model Evaluation ===
Accuracy: 0.8276
F1-Score: 0.8264


In [21]:
# Save pruned model
pruned_path = os.path.join(save_dir, "gesture_tcn_structured_pruned.pth")
torch.save(pruned_model.state_dict(), pruned_path)
log_info(f"Saved pruned model to {pruned_path}")

[22:40:32] INFO - Saved pruned model to checkpoints/gesture_tcn_structured_pruned.pth


## 8. Export to ONNX

In [22]:
import onnx
import onnxruntime as ort


def export_to_onnx(model, save_path, name):
    """Export PyTorch model to ONNX format."""
    model_cpu = model.cpu().eval()
    dummy_input = torch.randn(1, FEATURE_DIM, SEQ_LEN)

    try:
        torch.onnx.export(
            model_cpu,
            dummy_input,
            save_path,
            input_names=["input"],
            output_names=["logits"],
            dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
            opset_version=18,
        )
        onnx_model = onnx.load(save_path)
        onnx.checker.check_model(onnx_model)
        print(f"✓ {name}: Exported to {save_path}")
        print(f"  Size: {os.path.getsize(save_path) / (1024**2):.4f} MB")
        return True
    except Exception as e:
        print(f"✗ {name}: Failed - {e}")
        return False

In [23]:
log_info("Exporting models to ONNX ...")

# Export original model
original_onnx_path = os.path.join(save_dir, "gesture_tcn_original.onnx")
export_to_onnx(original_model, original_onnx_path, "Original")

# Export pruned model
pruned_onnx_path = os.path.join(save_dir, "gesture_tcn_pruned.onnx")
export_to_onnx(pruned_model, pruned_onnx_path, "Pruned")

[22:40:32] INFO - Exporting models to ONNX ...
✗ Original: Failed - No module named 'onnxscript'
✗ Pruned: Failed - No module named 'onnxscript'


False

## 9. ONNX Runtime INT8 Quantization (Method 3: Static Quantization with Calibration)

Use ONNX Runtime's post-training static quantization with calibration data.

In [24]:
from onnxruntime.quantization import (
    quantize_static,
    QuantFormat,
    QuantType,
    CalibrationDataReader,
)


class GestureCalibrationDataReader(CalibrationDataReader):
    """Calibration data reader for static quantization."""

    def __init__(self, data_loader, batch_size=1):
        self.data_loader = data_loader
        self.batch_size = batch_size
        self.iter = iter(data_loader)

    def get_next(self):
        try:
            x, _ = next(self.iter)
            return {"input": x.numpy()}
        except StopIteration:
            return None

    def rewind(self):
        self.iter = iter(self.data_loader)

In [25]:
log_info("Applying ONNX Runtime INT8 Static Quantization (Method 3) ...")

quantized_onnx_path = os.path.join(save_dir, "gesture_tcn_pruned_quantized.onnx")

try:
    # Create calibration data reader using test dataset
    calib_reader = GestureCalibrationDataReader(test_loader)

    # Apply static quantization
    quantize_static(
        model_input=pruned_onnx_path,
        model_output=quantized_onnx_path,
        calibration_data_reader=calib_reader,
        quant_format=QuantFormat.QDQ,
        per_channel=False,
        weight_type=QuantType.QInt8,
    )

    log_info("✓ Static quantization succeeded!")
    print(f"   Output: {quantized_onnx_path}")
    print(f"   Size: {os.path.getsize(quantized_onnx_path) / (1024**2):.4f} MB")
    quantization_success = True

except Exception as e:
    log_warn(f"Static quantization failed: {e}")
    quantization_success = False

[22:40:32] INFO - Applying ONNX Runtime INT8 Static Quantization (Method 3) ...
[22:40:32] WARNING - Static quantization failed: Unable to open proto file: checkpoints/gesture_tcn_pruned.onnx. Please check if it is a valid proto. 


## 10. Verify ONNX Models

In [26]:
def verify_onnx_inference(onnx_path, model_name):
    """Verify ONNX model produces same output as PyTorch."""
    try:
        session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        dummy_input = np.random.randn(1, FEATURE_DIM, SEQ_LEN).astype(np.float32)

        inputs = {session.get_inputs()[0].name: dummy_input}
        outputs = session.run(None, inputs)

        print(
            f"  ✓ {model_name}: ONNX inference verified, output shape: {outputs[0].shape}"
        )
        return True
    except Exception as e:
        print(f"  ✗ {model_name}: ONNX inference failed - {e}")
        return False

In [27]:
log_info("Verifying ONNX Inference ...")
verify_onnx_inference(original_onnx_path, "Original")
verify_onnx_inference(pruned_onnx_path, "Pruned")

if quantization_success:
    verify_onnx_inference(quantized_onnx_path, "Pruned + Quantized (INT8)")

[22:40:32] INFO - Verifying ONNX Inference ...
  ✗ Original: ONNX inference failed - [ONNXRuntimeError] : 3 : NO_SUCHFILE : Load model from checkpoints/gesture_tcn_original.onnx failed:Load model checkpoints/gesture_tcn_original.onnx failed. File doesn't exist
  ✗ Pruned: ONNX inference failed - [ONNXRuntimeError] : 3 : NO_SUCHFILE : Load model from checkpoints/gesture_tcn_pruned.onnx failed:Load model checkpoints/gesture_tcn_pruned.onnx failed. File doesn't exist


## 11. Evaluate Quantized ONNX Model

In [28]:
if quantization_success:
    log_info("Evaluating ONNX Quantized Model ...")

    ort_session = ort.InferenceSession(
        quantized_onnx_path, providers=["CPUExecutionProvider"]
    )

    all_preds = []
    all_labels = []

    for x, y in test_loader:
        for i in range(x.shape[0]):
            sample = x[i : i + 1].numpy()
            ort_inputs = {ort_session.get_inputs()[0].name: sample}
            ort_outputs = ort_session.run(None, ort_inputs)
            pred = np.argmax(ort_outputs[0], axis=1)[0]
            all_preds.append(pred)
            all_labels.append(y[i].item())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    onnx_accuracy = (all_preds == all_labels).mean()
    onnx_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

    print(f"\nONNX Quantized Model Accuracy: {onnx_accuracy:.4f}")
    print(f"ONNX Quantized Model F1-Score: {onnx_f1:.4f}")

    print(f"\nComparison with PyTorch Pruned Model:")
    print(
        f"  PyTorch Pruned: Accuracy {pruned_metrics['accuracy']:.4f}, F1 {pruned_metrics['f1_score']:.4f}"
    )
    print(f"  ONNX Quantized: Accuracy {onnx_accuracy:.4f}, F1 {onnx_f1:.4f}")

    print("\nClassification Report (ONNX Quantized):")
    print(
        classification_report(
            all_labels,
            all_preds,
            target_names=CLASS_NAMES,
            digits=4,
            zero_division=0,
        )
    )

## 12. Save Deployment Config

In [29]:
# Save deployment config with pruned channel info
config = {
    "model_name": "gesture_tcn",
    "class_names": CLASS_NAMES,
    "seq_len": SEQ_LEN,
    "feature_dim": FEATURE_DIM,
    "raw_dim": RAW_DIM,
    "num_classes": NUM_CLASSES,
    "num_landmarks": NUM_LANDMARKS,
    "normalize_mean": norm_stats["mean"].tolist(),
    "normalize_std": norm_stats["std"].tolist(),
    "pairs": PAIRS,
    "fingertip_ids": dataset_info["fingertip_ids"],
    "base_ids": dataset_info["base_ids"],
    "n_fingers": N_FINGERS,
    "finger_chains": FINGER_CHAINS,
    "prune_ratio": PRUNE_RATIO,
    "pruned_channels": pruned_channels,
    "original_params": count_parameters(original_model),
    "pruned_params": count_parameters(pruned_model),
}

config_path = os.path.join(save_dir, "config.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
log_info(f"Saved config to {config_path}")

[22:40:32] INFO - Saved config to checkpoints/config.json


In [30]:
# Summary
print("\n" + "=" * 80)
print("DEPLOYMENT SUMMARY")
print("=" * 80)

print("\n📊 Model Comparison:")
print("-" * 80)
print(f"{'Model':<30} {'Params':>10} {'Size':>12} {'Accuracy':>10}")
print("-" * 80)
print(
    f"{'Original (FP32)':<30} {count_parameters(original_model):>10,} {get_model_size_mb(original_model):>10.4f} MB {pruned_metrics['accuracy']:>10.2%}"
)
print(
    f"{'Structured Pruned (FP32)':<30} {count_parameters(pruned_model):>10,} {get_model_size_mb(pruned_model):>10.4f} MB {pruned_metrics['accuracy']:>10.2%}"
)

if quantization_success:
    onnx_size = os.path.getsize(quantized_onnx_path) / (1024**2)
    print(
        f"{'Pruned + Quantized (INT8 ONNX)':<30} {count_parameters(pruned_model):>10,} {onnx_size:>10.4f} MB {onnx_accuracy:>10.2%}"
    )

print("\n📁 Output Files:")
print("-" * 80)
print(f"  PyTorch Models:")
print(f"    - gesture_tcn_best.pth (original)")
print(f"    - gesture_tcn_structured_pruned.pth (pruned)")
print(f"\n  ONNX Models:")
print(f"    - gesture_tcn_original.onnx")
print(f"    - gesture_tcn_pruned.onnx")
if quantization_success:
    print(f"    - gesture_tcn_pruned_quantized.onnx (INT8)")


DEPLOYMENT SUMMARY

📊 Model Comparison:
--------------------------------------------------------------------------------
Model                              Params         Size   Accuracy
--------------------------------------------------------------------------------
Original (FP32)                    87,077     0.3365 MB     82.76%
Structured Pruned (FP32)           45,877     0.1781 MB     82.76%

📁 Output Files:
--------------------------------------------------------------------------------
  PyTorch Models:
    - gesture_tcn_best.pth (original)
    - gesture_tcn_structured_pruned.pth (pruned)

  ONNX Models:
    - gesture_tcn_original.onnx
    - gesture_tcn_pruned.onnx


## 13. Download Results

In [31]:
# List all saved files
print("\n=== Saved Files ===")
!ls -la {save_dir}/


=== Saved Files ===
total 804
drwxr-xr-x 3 xuranus xuranus   4096 Mar 24 22:40 .
drwxr-xr-x 6 xuranus xuranus   4096 Mar 24 22:36 ..
drwxr-xr-x 2 xuranus xuranus   4096 Mar 24 21:18 cache
-rw-r--r-- 1 xuranus xuranus   8399 Mar 24 22:40 config.json
-rw-r--r-- 1 xuranus xuranus  51839 Mar 24 22:27 confusion_matrix.png
-rw-r--r-- 1 xuranus xuranus   1055 Mar 24 21:52 dataset_info.json
-rw-r--r-- 1 xuranus xuranus 374937 Mar 24 22:27 gesture_tcn_best.pth
-rw-r--r-- 1 xuranus xuranus 209767 Mar 24 22:40 gesture_tcn_structured_pruned.pth
-rw-r--r-- 1 xuranus xuranus   3199 Mar 24 21:52 norm_stats.pt
-rw-r--r-- 1 xuranus xuranus 133154 Mar 24 22:27 training_curves.png
-rw-r--r-- 1 xuranus xuranus   8325 Mar 24 22:27 training_history.json


In [ ]:
# Download ONNX model (for Android deployment)
"""
from google.colab import files

if quantization_success:
    files.download(quantized_onnx_path)
files.download(os.path.join(save_dir, "config.json"))
"""

In [ ]:
# copy to Google Drive
"""
import shutil

drive_path = "/content/drive/MyDrive/checkpoints"
shutil.copytree("checkpoints", drive_path, dirs_exist_ok=True)
print(f"File copy to google drive: {drive_path}")
"""

## Summary

This notebook has:
1. Loaded the trained model
2. Applied **Structured Pruning** (30% channels removed)
3. **Fine-tuned** the pruned model to recover accuracy
4. Exported models to ONNX format
5. Applied **ONNX Runtime INT8 Static Quantization** (Method 3 with calibration)
6. Verified ONNX models and evaluated quantized model

**Key Results:**
- Structured Pruning: ~2x parameter reduction
- Fine-tuning: Recovers/improves accuracy
- INT8 Quantization: Additional size reduction for deployment

**Output files:**
- `gesture_tcn_original.onnx` - Original FP32 ONNX
- `gesture_tcn_pruned.onnx` - Pruned FP32 ONNX
- `gesture_tcn_pruned_quantized.onnx` - INT8 quantized ONNX
- `gesture_tcn_structured_pruned.pth` - Pruned PyTorch weights
- `config.json` - Model configuration